<a href="https://colab.research.google.com/github/robzulu26/AI-Study-Quiz-Assistant/blob/main/Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Ai-Powered Study & Quiz Assistant**


**CIT Capstone Project**

This project allows students to upload their own study material as a PDF and create practice quizzes using AI. Students can answer multiple-choice questions and receive their scores, correct answers, and explanations after completing the quiz. The goal is to help students review their material and prepare for tests.

**1. Setup and Import**

This section will install and import the libraries needed for this project.

In [ ]:
!pip install -q google-genai

**2. Connect to Gemini API**

In [ ]:
from google import genai
from google.colab import userdata
api_key = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)
print("Gemini client created successfuly!")

**3. Test Gemini API**

This section tests the connection by sending a simple request to Gemini.

In [ ]:
response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents="Say hello and confirm that the AI is working."
)
print(response.text)

**4.Upload Study Material**

This section allows students to upload a PDF containing their study notes. The PDF will be used to generate quiz questions.

In [ ]:
from google.colab import files
uploaded = files.upload()
pdf_filename = next(iter(uploaded))
print("PDF uploaded successfully:", pdf_filename)

**5.Extract Text from PDF**

This section reads the uploaded PDF and extracts the text so it can be used to generate quiz questions.

In [ ]:
!pip install -q pypdf
from pypdf import PdfReader
reader = PdfReader(pdf_filename)
study_material = ""
for page in reader.pages:
  study_material += (page.extract_text(extraction_mode="layout") or "") + "\n"
print("Text extracted successfully!")
print("Total characters:", len(study_material))
print("\nPreview of study material:")
print(study_material)

**6.Generate Quiz Questions**

This section uses Gemini to generate practice quiz questions based on the upoaded PDF study material.

In [ ]:
import json
from google.genai import types
prompt = f"""

Read the study material below and identify all the main topics or definitions.

Create one multiple-choice question for each main topic or definition.

Each question shoud have:
-Four answer choices (A,B,C,D)
-One correct answer
-A short explanation of the correct answer

Rules:
-Cover each main topic once.
-Do not repear questions.
-Use only information from the uploaded study material.
-Do not skip any clearly identidied definitions.

Study Material:
{study_material}
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type="application/json"
        )
)

quiz_questions = json.loads(response.text)
print("Quiz questions generated successfully!")
print("Total questions:", len(quiz_questions))


**7.Take the Quiz**

This section allows students to answer multiple-choice questions without seeing the correct answers.

In [ ]:
student_answers = []

for i, item in enumerate(quiz_questions, start=1):
    print(f"\nQuestion {i}: {item['question']}")

    choices = item["choices"]

    if isinstance(choices, dict):
        for letter, choice in choices.items():
            print(f"{letter}. {choice}")
    else:
        for letter, choice in zip(["A", "B", "C", "D"], choices):
            print(f"{letter}. {choice}")

    while True:
        answer = input("Your answer (A, B, C, or D): ").strip().upper()

        if answer in ["A", "B", "C", "D"]:
            student_answers.append(answer)
            break

        print("Please enter a valid answer (A, B, C, or D).")

print("\nQuiz completed! Your answers have been saved.")

**8.Scores and Feedback**

This section grades the completed quiz and shows the student's score, correct answers, and explanations.

In [ ]:
score = 0

for i, item in enumerate(quiz_questions):
  student_answer = student_answers[i]
  correct_answer = item["correct_answer"].strip().upper()

  print(f"\nQuestion {i + 1}: {item['question']}")
  print("Your answer:", student_answer)
  print("Correct answer:", correct_answer)

  if student_answer == correct_answer:
    score += 1
    print("Result: Correct!")
  else:
    print("Result: Incorrect.")

  print("Explanation:", item["explanation"])

total_questions = len(quiz_questions)
percentage = (score / total_questions) * 100

print("\n--- Final Quiz Results ---")
print(f"Score: {score}/{total_questions}")
print(f"Percentage: {percentage:.1f}%")
